# CareLoom: Agent Patterns — Larkspur Electronics

The first CareLoom agent could search and calculate, but it ran on its own — nothing paused
it, and nothing retried it if a downstream call hiccuped. This notebook covers the patterns
that make an agent safe to actually put in front of a support queue:

- ReAct-style agent loop
- `ToolNode`
- checkpointing with `InMemorySaver`
- `interrupt_before` for a human-approval pause
- error recovery with a retry policy

The LangGraph docs describe agents as dynamic workflows that use tools, and show `ToolNode` as the prebuilt node for tool execution. The checkpointer docs explain that checkpointing enables persistence and human-in-the-loop workflows, and the interrupts docs show how `interrupt_before` can pause a graph before a node runs. The fault-tolerance docs show retries and fallback logic.

## Learning goals

By the end of this notebook, you should be able to:

1. Build a small ReAct-style graph.
2. Route model output into tools with `ToolNode`.
3. Persist thread state with a checkpointer.
4. Pause before tool execution for approval.
5. Retry a failing node and recover cleanly.


## 1) Install packages


In [ ]:
%pip install -qU langgraph langchain langchain-core


## 2) Why this notebook does not need an API key

Larkspur's support lead wants to sign off on a new hire's first few tool calls — refund math and policy lookups, specifically — before trusting them to run unsupervised. To keep this lab runnable everywhere, and to isolate the approval/retry mechanics from anything an LLM might do differently between runs, the "trainee" here is a simple deterministic planner node standing in for that new hire, not a real tool-calling LLM.

The graph still demonstrates the same LangGraph patterns you'd use with a real tool-calling agent:
- a stateful graph
- a tool-using loop
- a checkpointer
- an approval pause
- retries for failure recovery


## 3) Import the LangGraph pieces


In [ ]:
import re
from typing import Literal
from typing_extensions import TypedDict

from langchain.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.runtime import Runtime
from langgraph.types import RetryPolicy


## 4) Define two small tools

We will use:
- a calculator, for refund and discount math
- a policy lookup tool, for Larkspur's refund/return/approval rules

`ToolNode` is the prebuilt LangGraph node that executes tool calls and handles state injection and error handling.


In [ ]:
@tool
def calculator(expression: str) -> str:
    '''Evaluate a basic math expression such as "4800 * 0.15".'''
    safe_globals = {"__builtins__": {}}
    safe_locals = {}
    try:
        value = eval(expression, safe_globals, safe_locals)
        return str(value)
    except Exception as e:
        return f"Calculator error: {e}"


@tool
def policy_lookup(query: str) -> str:
    '''Look up a Larkspur Electronics support policy from a small internal reference.'''
    policies = {
        "refund": "Refunds are approved within 7 business days once the returned item passes inspection.",
        "returns": "Unopened items can be returned within 14 days of delivery for a full refund.",
        "approval": "Refunds above ₹3,000 must be approved by a support lead before they are processed.",
    }

    text = query.lower()
    for key, value in policies.items():
        if key in text:
            return value

    return "No matching policy found."


## 5) Build a ReAct-style planner node

This node behaves like the "LLM" in a ReAct agent — a stand-in for a new support hire still under review:

- if the question needs a tool, it emits a tool call
- if a tool result is already present, it turns that result into a final answer
- if the question is simple, it answers directly

The LangGraph agent docs show the same overall loop: an LLM node decides whether to call a tool, a tool node executes the call, and the graph loops back to the LLM until no tool call is needed.

**A bug we hit while testing this cell live:** the first version of `_extract_expression` searched for `[\d\.\+\-\*/\(\)\s]+` — any run of digits, math symbols, *or whitespace*. Since a plain space character satisfies that class, `re.search` matched the very first space in the sentence (e.g. the one right after "What") and stopped there, because the next character was a letter. The "expression" it extracted was just `" "`, which strips down to an empty string and falls through to the `"1 + 1"` default — so every calculator question silently evaluated to `2`, regardless of what was actually asked. The fix is to require the match to *start* on a digit, so the search skips past the English words instead of latching onto the first stray space it finds.


In [ ]:
def _extract_expression(text: str) -> str:
    # Must start on a digit -- see the note above cell 5 on why the naive
    # "digits, symbols, or whitespace" character class matched the wrong thing.
    match = re.search(r"\d[\d\.\+\-\*/\(\)\s]*", text)
    if match:
        expr = match.group(0).strip()
        return expr if expr else "1 + 1"
    return "1 + 1"


def _pick_policy_query(text: str) -> str:
    return text.strip()


def react_planner(state: MessagesState) -> dict:
    messages = state["messages"]
    last = messages[-1]

    if isinstance(last, ToolMessage):
        return {
            "messages": [
                AIMessage(
                    content=f"Final answer from tool result: {last.content}"
                )
            ]
        }

    if isinstance(last, HumanMessage):
        text = last.content.lower()

        if any(word in text for word in ["refund", "return", "approval", "policy"]):
            query = _pick_policy_query(last.content)
            return {
                "messages": [
                    AIMessage(
                        content="I need to check the policy tool.",
                        tool_calls=[
                            {
                                "name": "policy_lookup",
                                "args": {"query": query},
                                "id": "policy_call_1",
                                "type": "tool_call",
                            }
                        ],
                    )
                ]
            }

        if any(sym in text for sym in ["+", "-", "*", "/", "calculate", "times", "multipl", "add"]):
            expr = _extract_expression(last.content)
            return {
                "messages": [
                    AIMessage(
                        content="I need to use the calculator.",
                        tool_calls=[
                            {
                                "name": "calculator",
                                "args": {"expression": expr},
                                "id": "calc_call_1",
                                "type": "tool_call",
                            }
                        ],
                    )
                ]
            }

        if "explain" in text or "how" in text:
            return {
                "messages": [
                    AIMessage(
                        content=(
                            "A short direct answer: this graph keeps shared state, uses a tool loop, "
                            "and can pause for a support lead's approval before certain actions run."
                        )
                    )
                ]
            }

    return {
        "messages": [
            AIMessage(
                content="I can answer directly without tools."
            )
        ]
    }


## 6) Route to tools or end

If the latest AI message contains tool calls, the graph goes to `tools`. Otherwise it stops.

This is the same conditional-edge idea shown in the LangGraph docs for agent loops.


In [ ]:
def should_continue(state: MessagesState) -> Literal["tools", END]:
    last_message = state["messages"][-1]
    if isinstance(last_message, AIMessage) and last_message.tool_calls:
        return "tools"
    return END


## 7) Build the graph with a checkpointer

LangGraph's persistence docs say checkpointers keep thread-scoped state so a run can be resumed later. The checkpointer docs also show `InMemorySaver` for experimentation.

`interrupt_before=["tools"]` is the approval gate: every tool call this trainee planner wants to make — a refund calculation, a policy lookup — pauses here first so a support lead can review it before it actually runs.


In [ ]:
builder = StateGraph(MessagesState)

builder.add_node("llm_call", react_planner)
builder.add_node("tools", ToolNode([calculator, policy_lookup]))

builder.add_edge(START, "llm_call")
builder.add_conditional_edges("llm_call", should_continue, ["tools", END])
builder.add_edge("tools", "llm_call")

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer, interrupt_before=["tools"])

print("Graph compiled with checkpointing and approval pause.")


## 8) Run a question that needs a tool

This question should route through the calculator: a customer's ₹4,800 order, and a 15% loyalty credit to work out.

We pass a `thread_id` so the checkpointer can store and resume the thread state. The interrupts docs say `thread_id` identifies the checkpointed thread.


In [ ]:
config = {
    "configurable": {
        "thread_id": "agent-demo-001"
    }
}

first = graph.invoke(
    {"messages": [HumanMessage(content="What is 4800 * 0.15 as a loyalty credit?")]},
    config=config,
)

first


## 9) Approve and resume

The graph is paused before the `tools` node because we compiled it with `interrupt_before=["tools"]`. The interrupts docs say you can then resume the graph by invoking it again with `None` and the same `thread_id`. Static interrupts are mostly for debugging, but they work well as a notebook approval checkpoint — or, here, as the support lead clicking "approve" on the trainee's proposed calculation.


In [ ]:
resumed = graph.invoke(None, config=config)
resumed


## 10) Follow-up turn that uses the same thread

Because the thread is checkpointed, a second invocation with the same `thread_id` can continue the conversation.

This is the short-term memory behavior described in the persistence docs.


In [ ]:
follow_up = graph.invoke(
    {"messages": [HumanMessage(content="How did you get that result?")]},
    config=config,
)

follow_up


## 11) A policy question

This question should route to the policy tool instead of the calculator.


In [ ]:
policy_run = graph.invoke(
    {"messages": [HumanMessage(content="What is the refund policy?")]},
    config={"configurable": {"thread_id": "agent-demo-002"}},
)

policy_run


## 12) Error recovery with retries

Larkspur's warehouse inventory service is occasionally flaky under load. The fault-tolerance docs show `RetryPolicy(max_attempts=3)` and explain that LangGraph can inspect retry state through `runtime.execution_info.node_attempt`.


In [ ]:
class RecoveryState(TypedDict):
    result: str


def flaky_node(state: RecoveryState, runtime: Runtime) -> RecoveryState:
    if runtime.execution_info.node_attempt == 1:
        raise ConnectionError("Temporary failure from the warehouse inventory service.")
    return {"result": f"Recovered on attempt {runtime.execution_info.node_attempt}"}


recovery_builder = StateGraph(RecoveryState)
recovery_builder.add_node(
    "flaky_node",
    flaky_node,
    retry_policy=RetryPolicy(max_attempts=3),
)
recovery_builder.add_edge(START, "flaky_node")
recovery_builder.add_edge("flaky_node", END)

recovery_graph = recovery_builder.compile()

recovery_graph.invoke({"result": ""})


## 13) What to remember

- `ToolNode` is the standard node for tool execution.
- `InMemorySaver` gives you thread-scoped checkpointing for a notebook lab.
- `interrupt_before` gives you a human-approval pause before a node runs.
- `RetryPolicy` handles transient failures cleanly.
- A ReAct-style loop is just: think → tool call → observe → answer.

## Key takeaways

- Tool use in LangGraph is modeled as a graph loop, not a black box.
- Checkpointing makes the conversation resumable.
- Interrupts let you pause before a sensitive action — here, before a trainee's refund calculation or policy lookup actually executes.
- Retry policies keep a node from failing permanently on a transient error, like a flaky inventory service.

### What's verified where

| Piece | Status |
|---|---|
| Everything in this notebook | Executed and verified in this sandbox — no LLM or external API involved, so there's nothing environment-dependent here. This is the one notebook in the module you can trust to behave identically on any machine. |
| `_extract_expression` regex | Live execution caught a real bug here (see the note in section 5) — a naive character class matched a stray space instead of skipping to the digits. Fixed and re-verified; the calculator now returns the actual computed value instead of silently defaulting to `2`. |

## References

- ToolNode: https://docs.langchain.com/oss/python/langgraph/workflows-agents#toolnode
- Checkpointers: https://docs.langchain.com/oss/python/langgraph/checkpointers
- Interrupts: https://docs.langchain.com/oss/python/langgraph/interrupts
- Fault tolerance: https://docs.langchain.com/oss/python/langgraph/fault-tolerance
